# 02 - Prepare Dataset

## Obstacle Detection Dataset

Notebook này thực hiện kiểm tra và chuẩn bị dataset YOLO
trước khi huấn luyện mô hình YOLOv8.

### Dataset

- Dataset: `Abtinzandi/Obstacle-Detection-Dataset-YOLO`
- Format: YOLO
- Số class: 25

### Nội dung

1. Kiểm tra đường dẫn dataset
2. Kiểm tra cấu trúc Train / Valid / Test
3. Thống kê số lượng ảnh
4. Thống kê số lượng label
5. Kiểm tra ảnh và label tương ứng
6. Kiểm tra `data.yaml`
7. Kiểm tra danh sách class
8. Kiểm tra định dạng YOLO label
9. Kiểm tra label hợp lệ
10. Hiển thị ảnh mẫu và bounding box

In [ ]:
from pathlib import Path
import yaml
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("..")

DATASET_DIR = PROJECT_DIR / "Obstacle-Detection-Dataset-YOLO"

TRAIN_IMAGE_DIR = DATASET_DIR / "train" / "images"
TRAIN_LABEL_DIR = DATASET_DIR / "train" / "labels"

VALID_IMAGE_DIR = DATASET_DIR / "valid" / "images"
VALID_LABEL_DIR = DATASET_DIR / "valid" / "labels"

TEST_IMAGE_DIR = DATASET_DIR / "test" / "images"
TEST_LABEL_DIR = DATASET_DIR / "test" / "labels"

DATA_YAML = DATASET_DIR / "data.yaml"

print("Dataset:", DATASET_DIR.resolve())

In [ ]:
if DATASET_DIR.exists():
    print("Dataset tồn tại.")
    print(DATASET_DIR.resolve())
else:
    print("Không tìm thấy dataset!")
    print("Hãy chạy notebook 01_download_dataset.ipynb trước.")

In [ ]:
required_dirs = [
    TRAIN_IMAGE_DIR,
    TRAIN_LABEL_DIR,
    VALID_IMAGE_DIR,
    VALID_LABEL_DIR,
    TEST_IMAGE_DIR,
    TEST_LABEL_DIR
]

print("Kiểm tra thư mục:\n")

for folder in required_dirs:
    if folder.exists():
        print(f"[OK] {folder}")
    else:
        print(f"[MISSING] {folder}")

Đếm số lượng ảnh


In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


def count_images(folder):
    return sum(
        1
        for file in folder.rglob("*")
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )


train_images = count_images(TRAIN_IMAGE_DIR)
valid_images = count_images(VALID_IMAGE_DIR)
test_images = count_images(TEST_IMAGE_DIR)

print("Train images:", train_images)
print("Valid images:", valid_images)
print("Test images :", test_images)

Đếm số lượng label

In [ ]:
def count_labels(folder):
    return len(list(folder.rglob("*.txt")))


train_labels = count_labels(TRAIN_LABEL_DIR)
valid_labels = count_labels(VALID_LABEL_DIR)
test_labels = count_labels(TEST_LABEL_DIR)

print("Train labels:", train_labels)
print("Valid labels:", valid_labels)
print("Test labels :", test_labels)

Kiểm tra ảnh và label có khớp

In [ ]:
def get_image_stems(folder):
    return {
        file.stem
        for file in folder.rglob("*")
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    }


def get_label_stems(folder):
    return {
        file.stem
        for file in folder.rglob("*.txt")
    }


def check_matching(image_dir, label_dir):

    image_stems = get_image_stems(image_dir)
    label_stems = get_label_stems(label_dir)

    missing_labels = image_stems - label_stems
    missing_images = label_stems - image_stems

    return image_stems, label_stems, missing_labels, missing_images

Kiểm tra từng tập


In [ ]:
splits = {
    "Train": (TRAIN_IMAGE_DIR, TRAIN_LABEL_DIR),
    "Valid": (VALID_IMAGE_DIR, VALID_LABEL_DIR),
    "Test": (TEST_IMAGE_DIR, TEST_LABEL_DIR)
}


for name, (image_dir, label_dir) in splits.items():

    images, labels, missing_labels, missing_images = check_matching(
        image_dir,
        label_dir
    )

    print(f"\n===== {name} =====")

    print("Images:", len(images))
    print("Labels:", len(labels))

    print("Missing labels:", len(missing_labels))
    print("Missing images:", len(missing_images))

Kiểm tra số class


In [ ]:
num_classes = data["nc"]
class_names = data["names"]

print("Number of classes:", num_classes)
print("Number of class names:", len(class_names))

Hiển thị danh sách class


In [ ]:
print("CLASS LIST")
print("=" * 30)

for class_id, class_name in enumerate(class_names):
    print(f"{class_id:2d} : {class_name}")

Kiểm tra đường dẫn trong data.yaml


In [ ]:
print("Train:", data.get("train"))
print("Validation:", data.get("val"))
print("Test:", data.get("test"))

In [ ]:
Hiển thị bounding box


In [1]:
def show_yolo_annotation(image_path, label_path, class_names):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.imshow(image)

    with open(label_path, "r", encoding="utf-8") as file:

        for line in file:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center, y_center, box_width, box_height = map(
                float,
                parts[1:]
            )

            x_center *= width
            y_center *= height
            box_width *= width
            box_height *= height

            x = x_center - box_width / 2
            y = y_center - box_height / 2

            rect = patches.Rectangle(
                (x, y),
                box_width,
                box_height,
                linewidth=2,
                fill=False
            )

            ax.add_patch(rect)

            ax.text(
                x,
                y,
                class_names[class_id],
                fontsize=10
            )

    ax.axis("off")
    plt.show()